# Rainfields Thunderstorm Tracking with PyFLEXTRKR

Tracks convective cells in Bureau of Meteorology rainfields3 ground reflectivity data using [PyFLEXTRKR](https://github.com/FlexTRKR/PyFLEXTRKR).

**Data path:** `/g/data/rq0/rainfields3/{radar_id}/{year}/gndrefl/{radar_id}_{date}.gndrefl.zip`

**Usage:** Edit the *Parameters* cell below, then `Kernel -> Restart & Run All`.  
**Kernel:** Select the `pyflextrkr` conda environment kernel.

In [15]:
# Parameters
radar_id   = 50              # BOM radar ID (integer)
date       = '20251101'      # Date to process (YYYYMMDD string)
output_dir = '/g/data/kl02/jss548/aint_testing/pyflextrckr'  # Output directory (required)

run_parallel = False   # True: use Dask LocalCluster (faster for large days)
n_workers    = 4       # Dask worker count (only used when run_parallel=True)
keep_staging = False   # True: keep temp staging dir for debugging

In [26]:
import os
import shutil
import tempfile
import zipfile
import logging
from pathlib import Path

import numpy as np
import xarray as xr
import yaml
from pyproj import CRS, Transformer

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s  %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('rainfields_tracking')
client = None  # Dask client placeholder


def preprocess_file(nc_path, out_path):
    """Read a rainfields gndrefl NC file, add a time dimension and compute lat/lon."""
    with xr.open_dataset(nc_path) as ds:
        timestamp = ds['valid_time'].values  # scalar numpy datetime64

        # Build Albers CRS from CF attributes stored in the 'proj' variable
        proj_attrs = dict(ds['proj'].attrs)
        if hasattr(proj_attrs.get('standard_parallel', None), '__iter__'):
            proj_attrs['standard_parallel'] = list(proj_attrs['standard_parallel'])
        crs_src   = CRS.from_cf(proj_attrs)
        crs_wgs84 = CRS.from_epsg(4326)
        transformer = Transformer.from_crs(crs_src, crs_wgs84, always_xy=True)

        # x/y are in km -- convert to metres for the Albers CRS
        x_m = ds['x'].values * 1000.0
        y_m = ds['y'].values * 1000.0
        xx, yy = np.meshgrid(x_m, y_m)
        lon2d, lat2d = transformer.transform(xx, yy)

        radar_lon_val = float(proj_attrs['longitude_of_central_meridian'])
        radar_lat_val = float(proj_attrs['latitude_of_projection_origin'])

        refl = ds['reflectivity'].expand_dims(dim={'time': [timestamp]})
        ds_out = xr.Dataset(
            {
                'reflectivity': refl,
                'lat': xr.DataArray(lat2d, dims=('y', 'x'), attrs={'units': 'degrees_north'}),
                'lon': xr.DataArray(lon2d, dims=('y', 'x'), attrs={'units': 'degrees_east'}),
                'radar_lat': xr.DataArray(
                    radar_lat_val,
                    attrs={'long_name': 'Radar latitude', 'units': 'degrees_north'},
                ),
                'radar_lon': xr.DataArray(
                    radar_lon_val,
                    attrs={'long_name': 'Radar longitude', 'units': 'degrees_east'},
                ),
            },
            coords={'x': ds['x'], 'y': ds['y'], 'time': [timestamp]},
            attrs=ds.attrs,
        )
        ds_out.to_netcdf(out_path)


def write_config(config_path, staging_dir, output_dir, radar_id, date, run_parallel):
    """Generate and write the PyFLEXTRKR YAML config for this run."""
    cfg = {
        # Pipeline stages
        'run_idfeature':   True,
        'run_advection':   True,
        'run_tracksingle': True,
        'run_gettracks':   True,
        'run_trackstats':  True,
        'run_mapfeature':  True,
        # Output paths
        'root_path':          str(output_dir),
        'tracking_path_name': 'tracking',
        'stats_path_name':    'stats',
        'pixel_path_name':    'pixeltracking',
        # Input data
        'clouddata_path': str(staging_dir) + '/',
        'databasename':   f'{radar_id}_',
        'time_format':    'yyyymodd_hhmmss',
        'input_format':   'netcdf',
        # Date range (yyyymmdd.hhmm)
        'startdate': f'{date}.0000',
        'enddate':   f'{date}.2359',
        # Feature type
        'feature_type': 'radar_cells',
        # Parallelism
        'run_parallel': 1 if run_parallel else 0,
        'nprocesses':   n_workers,
        'dask_tmp_dir': '/tmp',
        'timeout':      360,
        # Variable / coordinate names
        'reflectivity_varname': 'reflectivity',
        'ref_varname':          'dbz_comp',
        'x_dimname':            'x',
        'y_dimname':            'y',
        'x_coordname':          'x',
        'y_coordname':          'y',
        'lat_coordname':        'lat',
        'lon_coordname':        'lon',
        'radar_lat_varname':    'radar_lat',
        'radar_lon_varname':    'radar_lon',
        'time_dimname':         'time',
        'time_coordname':       'time',
        'is_3d': False,
        # Grid spacing in metres (0.5 km)
        'dx': 500,
        'dy': 500,
        # Data structure
        'datatimeresolution': 0.0833,  # 5-minute data in decimal hours
        'pixel_radius':       0.5,
        'area_method':        'cartesian',
        'radar_sensitivity':  0.0,
        # Steiner convective-cell classification
        'absConvThres':        60,
        'minZdiff':            10,
        'truncZconvThres':     55,
        'mindBZuse':           25,
        'dBZforMaxConvRadius': 60,
        'conv_rad_increment':  0.5,
        'conv_rad_start':      1.0,
        'bkg_refl_increment':  5,
        'convolve_method':     'fft',
        'dilate_method':       'orig',
        'expand_method':       'orig',
        'echotop_method':      'orig',
        'maxConvRadius':       5,
        'radii_expand':        [1, 2, 3, 4, 5],
        'weakEchoThres':       15,
        'bkgrndRadius':        11,
        'min_corearea':        4,
        'min_cellarea':        4,
        'echotop_gap':         4,
        'sfc_dz_min':          0,
        'sfc_dz_max':          3000,
        'return_diag':         False,
        # Advection
        'advection_field_threshold':  10,
        'advection_med_filt_len':     9,
        'advection_max_movement_mps': 60,
        'advection_mask_method':      'greater',
        'advection_buffer':           30,
        'advection_size_threshold':   10,
        'advection_tiles':            [1, 1],
        'advection_filename':         'advection_',
        # Tracking
        'timegap':                0.25,
        'nmaxlinks':              10,
        'othresh':                0.3,
        'maxnclouds':             1000,
        'duration_range':         [2, 100],
        'remove_shorttracks':     1,
        'trackstats_dense_netcdf': 1,
        'match_pixel_dt_thresh':  60.0,
        'fillval':                -9999,
        # Feature variable names
        'feature_varname':     'feature_number',
        'nfeature_varname':    'nfeatures',
        'featuresize_varname': 'npix_feature',
        # Track output dimensions
        'tracks_dimname':         'tracks',
        'times_dimname':          'times',
        'pixeltracking_filebase': 'celltracks_',
    }
    with open(config_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
    logger.info(f'Config written: {config_path}')

## 1. Extract zip to staging directory

In [5]:
year     = date[:4]
zip_path = Path(f'/g/data/rq0/rainfields3/{radar_id}/{year}/gndrefl/{radar_id}_{date}.gndrefl.zip')
logger.info(f'Source zip: {zip_path}')
assert zip_path.exists(), f'Zip not found: {zip_path}'

staging_dir      = Path(tempfile.mkdtemp(prefix=f'rainfields_{radar_id}_{date}_'))
raw_dir          = staging_dir / 'raw'
preprocessed_dir = staging_dir / 'preprocessed'
raw_dir.mkdir()
preprocessed_dir.mkdir()

with zipfile.ZipFile(zip_path) as zf:
    nc_names = [m for m in zf.namelist() if m.endswith('.gndrefl.nc')]
    for member in nc_names:
        with zf.open(member) as src, open(raw_dir / Path(member).name, 'wb') as dst:
            dst.write(src.read())

logger.info(f'Extracted {len(nc_names)} files to {raw_dir}')

22:26:12 INFO rainfields_tracking  Source zip: /g/data/rq0/rainfields3/50/2025/gndrefl/50_20251101.gndrefl.zip
22:26:14 INFO rainfields_tracking  Extracted 288 files to /scratch/kl02/jss548/tmp/rainfields_50_20251101_zlsf5fo_/raw


## 2. Preprocess: add time dimension and compute lat/lon

Rainfields NC files have no `time` dimension (only a scalar `valid_time`). PyFLEXTRKR requires a `time` coordinate.  
lat/lon are computed from the Albers projection stored in the file's `proj` variable.

In [6]:
raw_files = sorted(raw_dir.glob(f'{radar_id}_*.gndrefl.nc'))
logger.info(f'Preprocessing {len(raw_files)} files...')

for nc in raw_files:
    preprocess_file(nc, preprocessed_dir / nc.name)

logger.info('Preprocessing complete.')

22:26:16 INFO rainfields_tracking  Preprocessing 288 files...
22:28:01 INFO rainfields_tracking  Preprocessing complete.


## 3. Generate PyFLEXTRKR config and load it

In [27]:
os.makedirs(output_dir, exist_ok=True)
config_path = Path(output_dir) / f'config_{radar_id}_{date}.yml'
write_config(config_path, preprocessed_dir, output_dir, radar_id, date, run_parallel)

23:20:36 INFO rainfields_tracking  Config written: /g/data/kl02/jss548/aint_testing/pyflextrckr/config_50_20251101.yml


In [28]:
from pyflextrkr.ft_utilities import load_config

config = load_config(str(config_path))
logger.info(f'Config loaded. Output dirs created under: {output_dir}')

23:20:36 INFO rainfields_tracking  Config loaded. Output dirs created under: /g/data/kl02/jss548/aint_testing/pyflextrckr


## 4. (Optional) Set up Dask parallel processing

In [9]:
if run_parallel:
    from dask.distributed import Client, LocalCluster
    cluster = LocalCluster(n_workers=n_workers, threads_per_worker=1)
    client  = Client(cluster)
    logger.info(f'Dask dashboard: {client.dashboard_link}')
else:
    logger.info('Running in serial mode.')

22:28:20 INFO rainfields_tracking  Running in serial mode.


## 5. Run PyFLEXTRKR tracking pipeline

In [10]:
from pyflextrkr.idfeature_driver import idfeature_driver

logger.info('Step 1/5: Identifying convective cells...')
idfeature_driver(config)
logger.info('Cell identification complete.')

22:28:23 INFO rainfields_tracking  Step 1/5: Identifying convective cells...
22:28:23 INFO pyflextrkr.idfeature_driver  Identifying features from raw data
22:28:29 INFO pyflextrkr.idfeature_driver  Total number of files to process: 288
22:28:29 INFO pyflextrkr.idcells_reflectivity  Squeezed time dimension (size=1) from dataset
22:28:37 INFO pyflextrkr.idcells_reflectivity  Input data is 2D, skipping echo-top height calculations
22:28:38 INFO pyflextrkr.idcells_reflectivity  /g/data/kl02/jss548/aint_testing/pyflextrckr/tracking/cloudid_20251101_000000.nc
22:28:38 INFO pyflextrkr.idcells_reflectivity  Squeezed time dimension (size=1) from dataset
22:28:44 INFO pyflextrkr.idcells_reflectivity  Input data is 2D, skipping echo-top height calculations
22:28:44 INFO pyflextrkr.idcells_reflectivity  /g/data/kl02/jss548/aint_testing/pyflextrckr/tracking/cloudid_20251101_000500.nc
22:28:44 INFO pyflextrkr.idcells_reflectivity  Squeezed time dimension (size=1) from dataset
22:28:52 INFO pyflextrk

In [11]:
from pyflextrkr.tracksingle_driver import tracksingle_driver

logger.info('Step 2/5: Linking features between frames...')
tracksingle_driver(config)
logger.info('Single-step tracking complete.')

22:54:06 INFO rainfields_tracking  Step 2/5: Linking features between frames...
22:54:06 INFO pyflextrkr.tracksingle_driver  Tracking sequential pairs of idfeature files
22:54:06 INFO pyflextrkr.tracksingle_driver  Total number of files to process: 288
22:54:07 INFO pyflextrkr.tracksingle_drift  /g/data/kl02/jss548/aint_testing/pyflextrckr/tracking/track_20251101_000500.nc
22:54:07 INFO pyflextrkr.tracksingle_drift  /g/data/kl02/jss548/aint_testing/pyflextrckr/tracking/track_20251101_001000.nc
22:54:08 INFO pyflextrkr.tracksingle_drift  /g/data/kl02/jss548/aint_testing/pyflextrckr/tracking/track_20251101_001500.nc
22:54:08 INFO pyflextrkr.tracksingle_drift  /g/data/kl02/jss548/aint_testing/pyflextrckr/tracking/track_20251101_002000.nc
22:54:08 INFO pyflextrkr.tracksingle_drift  /g/data/kl02/jss548/aint_testing/pyflextrckr/tracking/track_20251101_002500.nc
22:54:08 INFO pyflextrkr.tracksingle_drift  /g/data/kl02/jss548/aint_testing/pyflextrckr/tracking/track_20251101_003000.nc
22:54:09 

In [20]:
from pyflextrkr.gettracks import gettracknumbers

logger.info('Step 3/5: Building full track chains...')
gettracknumbers(config)
logger.info('Track chain assignment complete.')

23:10:30 INFO rainfields_tracking  Step 3/5: Building full track chains...
23:10:30 INFO pyflextrkr.gettracks  Tracking features sequentially from single track files
23:10:30 INFO pyflextrkr.ft_utilities  Scanning track files to find maximum number of clouds...
23:10:30 INFO pyflextrkr.ft_utilities  Scanning 287 track files...
23:10:35 INFO pyflextrkr.ft_utilities  Maximum nclouds_ref: 45
23:10:35 INFO pyflextrkr.ft_utilities  Maximum nclouds_new: 45
23:10:35 INFO pyflextrkr.ft_utilities  Maximum nclouds across all files: 45
23:10:35 INFO pyflextrkr.gettracks  Total number of files to process: 287
23:10:35 INFO pyflextrkr.gettracks  track_20251101_000500.nc
23:10:36 INFO pyflextrkr.gettracks  track_20251101_001000.nc
23:10:36 INFO pyflextrkr.gettracks  track_20251101_001500.nc
23:10:36 INFO pyflextrkr.gettracks  track_20251101_002000.nc
23:10:36 INFO pyflextrkr.gettracks  track_20251101_002500.nc
23:10:36 INFO pyflextrkr.gettracks  track_20251101_003000.nc
23:10:36 INFO pyflextrkr.gett

In [29]:
from pyflextrkr.trackstats_driver import trackstats_driver

logger.info('Step 4/5: Computing track statistics...')
trackstats_driver(config)
logger.info('Track statistics complete.')

23:20:40 INFO rainfields_tracking  Step 4/5: Computing track statistics...
23:20:40 INFO pyflextrkr.trackstats_driver  Calculating track statistics
23:20:40 INFO pyflextrkr.trackstats_driver  Total number of files to process: 288
23:20:40 INFO pyflextrkr.trackstats_func  cloudid_20251101_000000.nc
23:20:40 INFO pyflextrkr.trackstats_func  cloudid_20251101_000500.nc
23:20:41 INFO pyflextrkr.trackstats_func  cloudid_20251101_001000.nc
23:20:41 INFO pyflextrkr.trackstats_func  cloudid_20251101_001500.nc
23:20:41 INFO pyflextrkr.trackstats_func  cloudid_20251101_002000.nc
23:20:41 INFO pyflextrkr.trackstats_func  cloudid_20251101_002500.nc
23:20:41 INFO pyflextrkr.trackstats_func  cloudid_20251101_003000.nc
23:20:41 INFO pyflextrkr.trackstats_func  cloudid_20251101_003500.nc
23:20:42 INFO pyflextrkr.trackstats_func  cloudid_20251101_004000.nc
23:20:42 INFO pyflextrkr.trackstats_func  cloudid_20251101_004500.nc
23:20:42 INFO pyflextrkr.trackstats_func  cloudid_20251101_005000.nc
23:20:42 IN

In [30]:
from pyflextrkr.mapfeature_driver import mapfeature_driver

logger.info('Step 5/5: Mapping tracked features to pixel files...')
mapfeature_driver(config)
logger.info('Feature mapping complete.')

23:21:30 INFO rainfields_tracking  Step 5/5: Mapping tracked features to pixel files...
23:21:30 INFO pyflextrkr.mapfeature_driver  Mapping tracked features to pixel-level files
23:21:30 INFO pyflextrkr.mapfeature_driver  Total number of files to process: 288
23:21:31 INFO pyflextrkr.mapfeature_func  /g/data/kl02/jss548/aint_testing/pyflextrckr/pixeltracking/20251101.0000_20251101.2359/celltracks_20251101_000000.nc
23:21:31 INFO pyflextrkr.mapfeature_func  /g/data/kl02/jss548/aint_testing/pyflextrckr/pixeltracking/20251101.0000_20251101.2359/celltracks_20251101_000500.nc
23:21:32 INFO pyflextrkr.mapfeature_func  /g/data/kl02/jss548/aint_testing/pyflextrckr/pixeltracking/20251101.0000_20251101.2359/celltracks_20251101_001000.nc
23:21:32 INFO pyflextrkr.mapfeature_func  /g/data/kl02/jss548/aint_testing/pyflextrckr/pixeltracking/20251101.0000_20251101.2359/celltracks_20251101_001500.nc
23:21:33 INFO pyflextrkr.mapfeature_func  /g/data/kl02/jss548/aint_testing/pyflextrckr/pixeltracking/202

## 6. Cleanup

In [31]:
if client is not None:
    client.close()

if keep_staging:
    logger.info(f'Staging dir kept at: {staging_dir}')
else:
    shutil.rmtree(staging_dir)
    logger.info('Staging dir removed.')

logger.info(f'Tracking complete. Output at: {output_dir}')

FileNotFoundError: [Errno 2] No such file or directory: PosixPath('/scratch/kl02/jss548/tmp/rainfields_50_20251101_zlsf5fo_')